# Perceptron Exercises

This notebook covers perceptron-based implementations for AND/OR logic, training a perceptron for the OR function, and using the delta rule with a sigmoid activation function.

## 1. Perceptron for AND and OR Functions

A perceptron computes a weighted sum of its inputs and applies an activation function (step function) to produce an output. Here, we implement 2-input AND and OR functions using suitable weights and thresholds.

### AND Function
- Weights: w1 = 0.6, w2 = 0.6
- Threshold (θ): 1.0

| x1 | x2 | Output (Y) |
|----|----|------------|
| 0  | 0  |     0      |
| 0  | 1  |     0      |
| 1  | 0  |     0      |
| 1  | 1  |     1      |

### OR Function
- Weights: w1 = 0.6, w2 = 0.6
- Threshold (θ): 0.5

| x1 | x2 | Output (Y) |
|----|----|------------|
| 0  | 0  |     0      |
| 0  | 1  |     1      |
| 1  | 0  |     1      |
| 1  | 1  |     1      |

In [103]:
import numpy as np

def perceptron(x1, x2, w1, w2, theta):
    X = x1 * w1 + x2 * w2
    return 1 if X >= theta else 0

# AND function
print('AND Function:')
for x1 in [0, 1]:
    for x2 in [0, 1]:
        y = perceptron(x1, x2, 0.6, 0.6, 1.0)
        print(f"x1={x1}, x2={x2} => Y={y}")

print('\nOR Function:')
for x1 in [0, 1]:
    for x2 in [0, 1]:
        y = perceptron(x1, x2, 0.6, 0.6, 0.5)
        print(f"x1={x1}, x2={x2} => Y={y}")

AND Function:
x1=0, x2=0 => Y=0
x1=0, x2=1 => Y=0
x1=1, x2=0 => Y=0
x1=1, x2=1 => Y=1

OR Function:
x1=0, x2=0 => Y=0
x1=0, x2=1 => Y=1
x1=1, x2=0 => Y=1
x1=1, x2=1 => Y=1


## 2. Training a Perceptron for 2-input OR Function

We use the Perceptron Learning Rule (Delta Rule) to train a perceptron for the OR function.
- Initial weights: w1 = 0.3, w2 = -0.1
- Threshold (θ): 0.2
- Learning rate (α): 0.1

The perceptron is trained on the OR truth table until it produces the correct output for all inputs.

In [104]:
# Perceptron training for OR function (one iteration example)
# Initial weights and parameters
w1, w2 = 0.3, -0.1
theta = 0.2
alpha = 0.1

# Training data for OR
X = np.array([[0,0],[0,1],[1,0],[1,1]])
Yd = np.array([0,1,1,1])

def step(x):
    return 1 if x >= 0 else 0

# Example: one iteration for input (1,0)
x1, x2 = 1, 0
d = 1
Xnet = x1*w1 + x2*w2 - theta
Y = step(Xnet)
e = d - Y

dw1 = alpha * x1 * e
w1_new = w1 + dw1

dw2 = alpha * x2 * e
w2_new = w2 + dw2

print(f"Net input: {Xnet:.2f}, Output: {Y}, Error: {e}")
print(f"Weight updates: Δw1={dw1}, Δw2={dw2}")
print(f"New weights: w1={w1_new}, w2={w2_new}")

Net input: 0.10, Output: 1, Error: 0
Weight updates: Δw1=0.0, Δw2=0.0
New weights: w1=0.3, w2=-0.1


## 3. Perceptron with Sigmoid Activation and Delta Rule

Now, consider a perceptron with two real-valued inputs and a sigmoid activation function. All initial weights and bias are 0.5. We show how the delta rule updates the weights for input x1 = 0.7, x2 = -0.6, desired output 1, and learning rate 0.1.

In [105]:
# Perceptron with sigmoid activation and delta rule
w1, w2, b = 0.5, 0.5, 0.5
alpha = 0.1
x1, x2 = 0.7, -0.6
d = 1

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Forward pass
Xnet = x1*w1 + x2*w2 + b
Y = sigmoid(Xnet)

# Error
error = d - Y

# Derivative of sigmoid at Y
f_prime = Y * (1 - Y)

# Weight and bias updates
Delta_w1 = alpha * error * x1 * f_prime
Delta_w2 = alpha * error * x2 * f_prime
Delta_b = alpha * error * 1 * f_prime

w1_new = w1 + Delta_w1
w2_new = w2 + Delta_w2
b_new = b + Delta_b

print(f"Net input: {Xnet:.3f}, Sigmoid output: {Y:.3f}")
print(f"Error: {error:.3f}, Derivative: {f_prime:.3f}")
print(f"Δw1={Delta_w1:.4f}, Δw2={Delta_w2:.4f}, Δb={Delta_b:.4f}")
print(f"New weights: w1={w1_new:.4f}, w2={w2_new:.4f}, bias={b_new:.4f}")

Net input: 0.550, Sigmoid output: 0.634
Error: 0.366, Derivative: 0.232
Δw1=0.0059, Δw2=-0.0051, Δb=0.0085
New weights: w1=0.5059, w2=0.4949, bias=0.5085


# 4. Predict the value of BMI (Body Mass Index) from the Height and Weight of a person using a perceptron

We use the `bmi.csv` dataset for training, ignoring the ‘gender’ column. The perceptron is implemented from scratch using the sigmoid activation function and trained using the delta rule.

**Workflow:**
- **Data Preprocessing:**  
    - Height and Weight are normalized (zero mean, unit variance).
    - BMI class labels are scaled to the range [0, 1] for sigmoid output compatibility.
- **Model:**  
    - Single-layer perceptron with sigmoid activation.
    - Weights (including bias) are initialized randomly.
- **Training:**  
    - The perceptron is trained using the delta rule (gradient descent) to minimize mean squared error.
- **Prediction:**  
    - After training, predictions are made and rescaled to the original BMI class labels.
    - Predicted values are rounded to the nearest integer class for evaluation.

In [109]:
import pandas as pd

# 1. Load and preprocess data
df = pd.read_csv('../bmi.csv')
X = df[['Height', 'Weight']].values.astype(float)
y = df['Index'].values.astype(float)

# Normalize inputs
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_norm = (X - X_mean) / X_std

# Scale outputs to [0,1]
y_scaled = y / 5.0

# 2. Define model: single-layer perceptron with sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(a):
    return a * (1 - a)

# 3. Training loop
np.random.seed(42)
w = np.random.randn(3) * 0.01  # [bias, w1, w2]
lr = 0.01
epochs = 1000

X_with_bias = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])  # prepend bias

for epoch in range(epochs):
    for i in range(X_with_bias.shape[0]):
        xi = X_with_bias[i]
        target = y_scaled[i]
        z = np.dot(w, xi)
        a = sigmoid(z)
        error = a - target
        grad = error * sigmoid_deriv(a)
        w -= lr * grad * xi
    # Print loss every 1000 epochs for inspection
    if (epoch + 1) % 1000 == 0 or epoch == 0:
        loss = np.mean((sigmoid(np.dot(X_with_bias, w)) - y_scaled) ** 2)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")
        

# 4. Inference: predict and rescale
z_pred = np.dot(X_with_bias, w)
y_pred_scaled = sigmoid(z_pred)
y_pred = y_pred_scaled * 5.0  # Rescale back to BMI classes

# 5. Optional: round predictions to nearest class label if reporting accuracy
y_pred_class = np.round(y_pred).astype(int)

# 6. Print sample predictions for inspection
for i in range(10):
    print(f"True: {int(y[i])}, Predicted: {y_pred_class[i]} (Raw: {y_pred[i]:.2f})")

Epoch 1/1000, Loss: 0.0797
Epoch 1000/1000, Loss: 0.0051
True: 4, Predicted: 4 (Raw: 3.67)
True: 2, Predicted: 2 (Raw: 2.22)
True: 4, Predicted: 4 (Raw: 3.81)
True: 3, Predicted: 3 (Raw: 2.92)
True: 3, Predicted: 3 (Raw: 3.11)
True: 3, Predicted: 3 (Raw: 3.29)
True: 5, Predicted: 4 (Raw: 4.50)
True: 5, Predicted: 5 (Raw: 4.72)
True: 3, Predicted: 3 (Raw: 3.34)
True: 4, Predicted: 4 (Raw: 4.18)
